In [ ]:
!pip install -q pdfplumber tqdm

In [ ]:


import pdfplumber
import sqlite3
import pandas as pd
import re
import os
from tqdm import tqdm

PDF_PATH = "/content/co-ursi-key-regultry-parmtrs-pwr-utiltis-dt180526.pdf"

DB_PATH = "pdf_index.db"
CSV_PATH = "page_index.csv"


TABLE_REGEX = re.compile(
    r"(TABLE[- ]?\d+(?:\([A-Z]\))?\s*:\s*[^\n]+)",
    flags=re.IGNORECASE
)


def detect_tables(text):

    matches = TABLE_REGEX.findall(text)

    cleaned = []

    for m in matches:

        m = re.sub(r"\s+", " ", m).strip()

        if len(m) > 8:
            cleaned.append(m)

    return sorted(list(set(cleaned)))


if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()


cur.execute("""
CREATE TABLE page_index (

    page_number INTEGER PRIMARY KEY,

    page_text TEXT,

    table_titles TEXT,

    contains_table INTEGER,

    text_length INTEGER
)
""")


cur.execute("""
CREATE VIRTUAL TABLE page_search
USING fts5(

    page_text,

    table_titles,

    content=''

)
""")

conn.commit()


rows = []

with pdfplumber.open(PDF_PATH) as pdf:

    total_pages = len(pdf.pages)

    print("="*70)
    print("PDF INFORMATION")
    print("="*70)

    print("Total Pages :", total_pages)

    for page_idx in tqdm(range(total_pages)):

        page = pdf.pages[page_idx]

        try:
            text = page.extract_text() or ""
        except:
            text = ""

        tables = detect_tables(text)

        table_text = " | ".join(tables)

        contains_table = 1 if len(tables) > 0 else 0

        rows.append({

            "page_number": page_idx + 1,
            "table_titles": table_text,
            "contains_table": contains_table,
            "text_length": len(text),
            "page_text": text
        })


df = pd.DataFrame(rows)


for _, row in df.iterrows():

    cur.execute(
        """
        INSERT INTO page_index
        VALUES(?,?,?,?,?)
        """,
        (
            int(row.page_number),
            row.page_text,
            row.table_titles,
            int(row.contains_table),
            int(row.text_length)
        )
    )

    cur.execute(
        """
        INSERT INTO page_search
        (
            page_text,
            table_titles
        )
        VALUES (?,?)
        """,
        (
            row.page_text,
            row.table_titles
        )
    )

conn.commit()


df.to_csv(CSV_PATH, index=False)


print()
print("="*70)
print("INDEX CREATED")
print("="*70)

print("Pages Indexed :", len(df))
print("Pages With Tables :", int(df.contains_table.sum()))


print()
print("="*70)
print("TABLE PAGES")
print("="*70)

display(

    df[
        df.contains_table == 1
    ][
        [
            "page_number",
            "table_titles"
        ]
    ].head(50)

)


print()
print("="*70)
print("FTS5 SEARCH TESTS")
print("="*70)

queries = [

    "open access",
    "cross subsidy surcharge",
    "wheeling charge",
    "transmission charge",
    "banking charges"

]

for q in queries:

    results = cur.execute(
        """
        SELECT rowid
        FROM page_search
        WHERE page_search MATCH ?
        LIMIT 20
        """,
        (q,)
    ).fetchall()

    pages = [r[0] for r in results]

    print()
    print("QUERY :", q)
    print("PAGES :", pages)


print()
print("="*70)
print("OPEN ACCESS REGION")
print("="*70)

display(

    df[
        (df.page_number >= 49)
        &
        (df.page_number <= 65)
    ][
        [
            "page_number",
            "table_titles",
            "contains_table"
        ]
    ]

)


print()
print("="*70)
print("FILES")
print("="*70)

print(
    "SQLite:",
    round(
        os.path.getsize(DB_PATH)/(1024*1024),
        2
    ),
    "MB"
)

print(
    "CSV:",
    round(
        os.path.getsize(CSV_PATH)/(1024*1024),
        2
    ),
    "MB"
)

conn.close()

PDF INFORMATION
Total Pages : 294


100%|██████████| 294/294 [01:06<00:00,  4.45it/s]


INDEX CREATED
Pages Indexed : 294
Pages With Tables : 32

TABLE PAGES


,page_number,table_titles
4,5,TABLE-1(A) : TARIFF ORDER FOR FY-27 AND TRUE U...
5,6,TABLE-1: TRAJECTORY FOR AT&C LOSS UNDER RDSS ....
31,32,"Table-2(a): Domestic Category (3 Kilo Watt, 10..."
33,34,"Table-2(b): Domestic Category (3 Kilo Watt, 50..."
35,36,Table-2(c): Non-Domestic Category (5 Kilo Watt...
37,38,Table-2(d): Agriculture Category (7.5 Kilo Wat...
39,40,Table-2(e): Small Industrial Category (10 Kilo...
41,42,Table-2(f): Medium Industrial Category (50 Kil...
43,44,Table-2(g): Large Industrial Category (1000 Ki...
45,46,Table-3: Cross Subsidy and Consumption Trend f...



FTS5 SEARCH TESTS

QUERY : open access
PAGES : [5, 14, 17, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 74]

QUERY : cross subsidy surcharge
PAGES : [5, 14, 50, 56, 157, 162, 243, 251, 265, 290]

QUERY : wheeling charge
PAGES : [5, 14, 59]

QUERY : transmission charge
PAGES : [5, 14, 62, 145, 153, 175, 194, 252, 283]

QUERY : banking charges
PAGES : [5, 50, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75]

OPEN ACCESS REGION


,page_number,table_titles,contains_table
48,49,,0
49,50,Table-5(a): Cross Subsidy Surcharge | Table-5:...,1
50,51,,0
51,52,,0
52,53,,0
53,54,,0
54,55,,0
55,56,,0
56,57,Table-5(b): Additional Surcharge,1
57,58,,0



FILES
SQLite: 1.1 MB
CSV: 0.67 MB


In [ ]:


import pandas as pd
import json

CSV_PATH = "page_index.csv"

df = pd.read_csv(CSV_PATH)


table_pages = df[
    (df["contains_table"] == 1) &
    (df["page_number"] >= 20)
].copy()

table_pages = table_pages.sort_values(
    "page_number"
)



catalog = []

for _, row in table_pages.iterrows():

    page = int(row["page_number"])

    titles = str(row["table_titles"])

    for title in titles.split("|"):

        title = title.strip()

        if len(title) < 5:
            continue

        catalog.append({
            "page": page,
            "title": title
        })

catalog_df = pd.DataFrame(catalog)


anchors = sorted(
    table_pages["page_number"].tolist()
)

def get_section(start_page):

    idx = anchors.index(start_page)

    if idx == len(anchors) - 1:

        end_page = int(df.page_number.max())

    else:

        end_page = anchors[idx + 1] - 1

    return list(
        range(start_page, end_page + 1)
    )


SEARCH_MAP = {

    "open access":
        "open access",

    "cross subsidy surcharge":
        "cross subsidy surcharge",

    "additional surcharge":
        "additional surcharge",

    "wheeling charge":
        "wheeling charge",

    "transmission charge":
        "transmission charge",

    "banking charges":
        "banking charges",

    "tod":
        "tod",

    "return on equity":
        "return on equity",

    "pending petitions":
        "pending petitions",

    "reliability":
        "reliability"
}


def find_relevant_pages(query):

    q = query.lower()

    keyword = None

    for k in SEARCH_MAP:

        if k in q:
            keyword = SEARCH_MAP[k]
            break

    if keyword is None:

        return {
            "success": False,
            "message": "keyword not found"
        }

    hits = catalog_df[
        catalog_df["title"]
        .str.contains(
            keyword,
            case=False,
            na=False
        )
    ]

    if len(hits) == 0:

        return {
            "success": False,
            "message": "table not found"
        }

    hit = hits.iloc[0]

    page = int(hit["page"])

    pages = get_section(page)

    return {

        "success": True,

        "query": query,

        "matched_table":
            hit["title"],

        "section_start":
            pages[0],

        "section_end":
            pages[-1],

        "pages":
            pages
    }


tests = [

    "Extract Open Access Charges",

    "Extract Cross Subsidy Surcharge",

    "Extract Additional Surcharge",

    "Extract Wheeling Charge",

    "Extract Transmission Charge",

    "Extract Banking Charges"
]

print("="*70)
print("PHASE 2.3 TESTS")
print("="*70)

for t in tests:

    print()
    print(
        json.dumps(
            find_relevant_pages(t),
            indent=2
        )
    )

PHASE 2.3 TESTS

{
  "success": true,
  "query": "Extract Open Access Charges",
  "matched_table": "Table-5: Open Access Charges",
  "section_start": 50,
  "section_end": 56,
  "pages": [
    50,
    51,
    52,
    53,
    54,
    55,
    56
  ]
}

{
  "success": true,
  "query": "Extract Cross Subsidy Surcharge",
  "matched_table": "Table-5(a): Cross Subsidy Surcharge",
  "section_start": 50,
  "section_end": 56,
  "pages": [
    50,
    51,
    52,
    53,
    54,
    55,
    56
  ]
}

{
  "success": true,
  "query": "Extract Additional Surcharge",
  "matched_table": "Table-5(b): Additional Surcharge",
  "section_start": 57,
  "section_end": 58,
  "pages": [
    57,
    58
  ]
}

{
  "success": true,
  "query": "Extract Wheeling Charge",
  "matched_table": "Table-5(c): Wheeling Charge",
  "section_start": 59,
  "section_end": 61,
  "pages": [
    59,
    60,
    61
  ]
}

{
  "success": true,
  "query": "Extract Transmission Charge",
  "matched_table": "Table-5(d): Transmission Char

In [ ]:
# import pandas as pd

# df = pd.read_csv("page_index.csv")

# for _, row in df[df["contains_table"] == 1].iterrows():

#     print("\nPAGE:", row["page_number"])
#     print(row["table_titles"][:500])
#     print("-"*80)


PAGE: 5
TABLE-1(A) : TARIFF ORDER FOR FY-27 AND TRUE UP ORDER FOR FY-25 FOR DISTRIBUTION UTILITIES .................... 22 | TABLE-1(B) : ISSUANCE OF MULTI YEAR TARIFF (MYT) ORDER ................................................................................. 29 | TABLE-10(A) : AUTOMATIC ADJUSTMENT OF FUEL AND POWER PURCHASE COST- COMPLIANCE STATUS AT SERC/JERC | TABLE-10(B) : AUTOMATIC ADJUSTMENT OF FUEL AND POWER PURCHASE COST- COMPLIANCE STATUS AT DISCOMS | TABLE-11: REVENUE GAP CREATION AND LIQUID
--------------------------------------------------------------------------------

PAGE: 6
TABLE-1: TRAJECTORY FOR AT&C LOSS UNDER RDSS ............................................................................................ 284 | TABLE-2: TRAJECTORY FOR ACS-ARR GAP UNDER RDSS ....................................................................................... 286
--------------------------------------------------------------------------------

PAGE: 32
Table-2(a): Domestic Categ

In [ ]:
# import pandas as pd

# df = pd.read_csv("page_index.csv")

# table_pages = df[df["contains_table"] == 1]

# table_pages = table_pages[
#     table_pages["page_number"] >= 20
# ]

# print(
#     table_pages[
#         ["page_number","table_titles"]
#     ]
# )

     page_number                                       table_titles
31            32  Table-2(a): Domestic Category (3 Kilo Watt, 10...
33            34  Table-2(b): Domestic Category (3 Kilo Watt, 50...
35            36  Table-2(c): Non-Domestic Category (5 Kilo Watt...
37            38  Table-2(d): Agriculture Category (7.5 Kilo Wat...
39            40  Table-2(e): Small Industrial Category (10 Kilo...
41            42  Table-2(f): Medium Industrial Category (50 Kil...
43            44  Table-2(g): Large Industrial Category (1000 Ki...
45            46  Table-3: Cross Subsidy and Consumption Trend f...
47            48  Table-4: Cross Subsidy and Consumption Trend f...
49            50  Table-5(a): Cross Subsidy Surcharge | Table-5:...
56            57                   Table-5(b): Additional Surcharge
58            59                        Table-5(c): Wheeling Charge
61            62                    Table-5(d): Transmission Charge
63            64                        Table-5(

In [ ]:
!apt-get -qq install poppler-utils > /dev/null
!pip install -q pdf2image opencv-python-headless pillow pandas

In [ ]:

# !apt-get -qq install poppler-utils > /dev/null
# !pip install -q pdf2image opencv-python-headless pillow pandas

import cv2
import numpy as np
import pandas as pd

from pdf2image import convert_from_path


PDF_PATH = "/content/co-ursi-key-regultry-parmtrs-pwr-utiltis-dt180526.pdf"

# Candidate pages from Phase 2
CANDIDATE_PAGES = [
    50,51,52,53,54,55,56
]

print("="*70)
print("RENDERING PAGES")
print("="*70)

images = convert_from_path(

    PDF_PATH,

    dpi=200,

    first_page=min(CANDIDATE_PAGES),

    last_page=max(CANDIDATE_PAGES)
)


def detect_table(image):

    gray = cv2.cvtColor(
        np.array(image),
        cv2.COLOR_RGB2GRAY
    )

    thresh = cv2.adaptiveThreshold(

        gray,

        255,

        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,

        cv2.THRESH_BINARY_INV,

        15,

        4
    )


    horizontal_kernel = cv2.getStructuringElement(

        cv2.MORPH_RECT,

        (40,1)
    )

    horizontal = cv2.morphologyEx(

        thresh,

        cv2.MORPH_OPEN,

        horizontal_kernel
    )



    vertical_kernel = cv2.getStructuringElement(

        cv2.MORPH_RECT,

        (1,40)
    )

    vertical = cv2.morphologyEx(

        thresh,

        cv2.MORPH_OPEN,

        vertical_kernel
    )



    h_cnts, _ = cv2.findContours(

        horizontal,

        cv2.RETR_EXTERNAL,

        cv2.CHAIN_APPROX_SIMPLE
    )

    v_cnts, _ = cv2.findContours(

        vertical,

        cv2.RETR_EXTERNAL,

        cv2.CHAIN_APPROX_SIMPLE
    )



    grid = cv2.add(
        horizontal,
        vertical
    )

    contours, _ = cv2.findContours(

        grid,

        cv2.RETR_EXTERNAL,

        cv2.CHAIN_APPROX_SIMPLE
    )

    rect_count = 0

    for c in contours:

        x,y,w,h = cv2.boundingRect(c)

        area = w*h

        if area > 20000:
            rect_count += 1


    score = (

        min(len(h_cnts),50)/50 * 0.4 +

        min(len(v_cnts),50)/50 * 0.4 +

        min(rect_count,10)/10 * 0.2
    )

    contains_table = score > 0.20

    return {

        "horizontal_lines":
            len(h_cnts),

        "vertical_lines":
            len(v_cnts),

        "large_rectangles":
            rect_count,

        "table_score":
            round(score,3),

        "contains_table":
            contains_table
    }


results = []

for page_num, image in zip(
    CANDIDATE_PAGES,
    images
):

    result = detect_table(image)

    result["page"] = page_num

    results.append(result)



results_df = pd.DataFrame(results)

print()
print("="*70)
print("TABLE DETECTION RESULTS")
print("="*70)

display(results_df)


table_pages = results_df[
    results_df["contains_table"] == True
]

print()
print("="*70)
print("PAGES SENT TO OCR")
print("="*70)

print(
    table_pages["page"].tolist()
)

RENDERING PAGES

TABLE DETECTION RESULTS


,horizontal_lines,vertical_lines,large_rectangles,table_score,contains_table,page
0,122,12,1,0.516,True,50
1,150,37,4,0.776,True,51
2,166,42,6,0.856,True,52
3,116,19,4,0.632,True,53
4,145,29,7,0.772,True,54
5,118,25,5,0.700,True,55
6,24,4,1,0.244,True,56



PAGES SENT TO OCR
[50, 51, 52, 53, 54, 55, 56]


In [ ]:
OUTPUT_CSV_PATH = "table_detection_results.csv"
results_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Table detection results saved to {OUTPUT_CSV_PATH}")

Table detection results saved to table_detection_results.csv


In [ ]:

import pdfplumber
import pandas as pd
import json


PDF_PATH = "/content/co-ursi-key-regultry-parmtrs-pwr-utiltis-dt180526.pdf"

CANDIDATE_PAGES = [
    50,51,52,53,54,55
]


results = []

with pdfplumber.open(PDF_PATH) as pdf:

    for page_num in CANDIDATE_PAGES:

        page = pdf.pages[page_num - 1]

        try:

            tables = page.extract_tables()

        except Exception as e:

            print(
                f"Page {page_num} failed:",
                e
            )

            continue

        page_result = {

            "page": page_num,

            "extractor": "pdfplumber",

            "table_count": len(tables),

            "tables": []
        }



        for idx, table in enumerate(tables):

            cleaned = []

            for row in table:

                if row is None:
                    continue

                cleaned_row = []

                for cell in row:

                    if cell is None:
                        cleaned_row.append("")
                    else:
                        cleaned_row.append(
                            str(cell).strip()
                        )

                cleaned.append(cleaned_row)

            page_result["tables"].append({

                "table_id": idx + 1,

                "rows": cleaned,

                "n_rows": len(cleaned),

                "n_cols": max(
                    [len(r) for r in cleaned],
                    default=0
                )
            })

        results.append(page_result)


print("="*80)
print("TABLE EXTRACTION SUMMARY")
print("="*80)

for page in results:

    print()

    print(
        f"Page {page['page']}"
    )

    print(
        f"Tables: {page['table_count']}"
    )

    for t in page["tables"]:

        print(
            f"  Table {t['table_id']} "
            f"({t['n_rows']} x {t['n_cols']})"
        )

with open(
    "phase4_tables.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )

print()
print("="*80)
print("SAVED")
print("="*80)
print("phase4_tables.json")

TABLE EXTRACTION SUMMARY

Page 50
Tables: 1
  Table 1 (25 x 12)

Page 51
Tables: 1
  Table 1 (57 x 16)

Page 52
Tables: 1
  Table 1 (58 x 21)

Page 53
Tables: 1
  Table 1 (56 x 14)

Page 54
Tables: 1
  Table 1 (54 x 21)

Page 55
Tables: 1
  Table 1 (58 x 19)

SAVED
phase4_tables.json


In [ ]:
import json

with open(
    "phase4_tables.json",
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

for page in data:

    print("\n")
    print("="*100)
    print("PAGE", page["page"])
    print("="*100)

    for table in page["tables"]:

        print("\nTABLE", table["table_id"])

        df = pd.DataFrame(
            table["rows"]
        )

        display(df.head(15))



PAGE 50

TABLE 1


,0,1,2,3,4,5,6,7,8,9,10,11
0,,आं(cid:364) (cid:366)देश / Andhra Pradesh,,,2026-27,,,,,,,
1,,Category,,,APSPDCL,,,APEPDCL,,,APCPDCL,
2,,HT Category at 11 kV,,,,,,,,,,
3,"HT I(B) Townships, Colonies, Gated\nCommunitie...",,,0.97,,,0.89,,,1.12,,
4,HT II (A) Commercial& Others,,,2.09,,,2.20,,,2.32,,
5,HT II (A) Function Halls/ Auditoriums,,,1.99,,,1.99,,,1.99,,
6,HT II (B) Start-up power,,,1.99,,,1.99,,,-,,
7,HT III(A) Industry,,,1.89,,,1.71,,,1.85,,
8,HT III(B) Seasonal Industries (Off Season),,,2.38,,,1.76,,,1.74,,
9,HT III (C ) Energy Intensive Industries,,,1.58,,,-,,,1.75,,




PAGE 51

TABLE 1


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,HT II(B) Start Up Power,,,1.99,,,1.99,,,,,,,1.99,,
1,HT III(A) Industry,,,1.37,,,1.44,,,,,,,1.40,,
2,HT III(C ) Energy Intensive Industry,,,0.31,,,0.51,,,,,,,-,,
3,HT IV(D) Railway Traction,,,1.63,,,1.59,,,,,,,1.56,,
4,HT V (E ),,,1.43,,,-,,,,,,,1.43,,
5,,220 kV,,,,,,,,,,,,,,
6,HT II(A) Commercial & Others,,,-,,,1.66,,,,,,,-,,
7,HT II(B) Start up Power,,,1.99,,,1.99,,,,,,,-,,
8,HT III(A) Industry,,,1.57,,,1.37,,,,,,,1.96,,
9,HT III (C ) Energy Intensive Industries,,,-,,,1.57,,,,,,,-,,




PAGE 52

TABLE 1


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,LT-3b,,,3.05,,3.05,,3.05,,,3.05,,3.05,
1,LT-4a,,,0.49,,0.49,,0.49,,,0.49,,0.49,
2,LT-4b,,,1.07,,1.07,,1.07,,,1.07,,1.07,
3,LT-4c,,,1.85,,1.85,,1.85,,,1.85,,1.85,
4,LT-5 Industrial,,,1.16,,1.16,,1.16,,,1.16,,1.16,
5,LT-7 Temporary Supply,,,3.19,,3.19,,3.19,,,3.19,,3.19,
6,,केरल / Kerala,,,2026-27,,,,,,,,,
7,,Category,,,KSEBL,,,,,,,,,
8,EHT- I,,,1.41,,,,,,,,,,
9,EHT- II,,,1.34,,,,,,,,,,




PAGE 54

TABLE 1


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,EHT Level,,,,,,,,,,...,,,,,,,,,,
1,HT & EHT Ferro Alloys Industrial,,,NIL,,,,,,,...,,,,,,,,,,
2,,ओिडशा / Odisha,,,2026-27,,,,,,...,,,,,,,,,,
3,,Category,,,TPCODL,,,TPNODL,,,...,,,,,,TPSODL,,,TPWODL,
4,EHT,,,,1.52,,,1.43,,,...,,,,,,1.18,,,2.50,
5,HT,,,,0.56,,,0.07,,,...,,,,,,0.17,,,1.06,
6,LT,,,,1.13,,,0.34,,,...,,,,,,0.24,,,1.40,
7,,पुडुचेरी / Puducherry,,,2026-27,,,,,,...,,,,,,,,,,
8,,Category,,,Puducherry Electricity Department (PED),,,,,,...,,,,,,,,,,
9,LT Level,,,NIL,,,,,,,...,,,,,,,,,,




PAGE 55

TABLE 1


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,,HT Category 132kV,,,,,,,,,,,,,,,,,
1,HT-I Industry,,,1.62,,,,,,,,1.69,,,,,,,
2,HT-I(B)Ferro Alloys,,,-,,,,,,,,-,,,,,,,
3,HT-II Others (Commercial),,,2.34,,,,,,,,3.77,,,,,,,
4,"HT-III Airports, Bus & Rly Stations",,,1.71,,,,,,,,-,,,,,,,
5,HT-IV(A) Irrigation & Agriculture,,,1.60,,,,,,,,1.92,,,,,,,
6,HT-IV(B) CPWS,,,0.76,,,,,,,,0.58,,,,,,,
7,HT-V(A) Railway Traction,,,1.34,,,,,,,,1.01,,,,,,,
8,HT-V(B)HMR,,,0.53,,,,,,,,-,,,,,,,
9,HT-VI Townships & Residential Colonies,,,-,,,,,,,,1.74,,,,,,,


In [ ]:

import json
import pandas as pd



with open(
    "phase4_tables.json",
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)



def normalize_table(rows):

    if len(rows) == 0:
        return rows

    df = pd.DataFrame(rows)


    keep_cols = []

    for col in df.columns:

        values = (
            df[col]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        non_empty = (values != "").sum()

        if non_empty > 0:
            keep_cols.append(col)

    df = df[keep_cols]


    keep_rows = []

    for idx in df.index:

        vals = (
            df.loc[idx]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        if (vals != "").sum() > 0:
            keep_rows.append(idx)

    df = df.loc[keep_rows]

    return df.reset_index(
        drop=True
    ).values.tolist()


normalized = []

for page in data:

    page_out = {

        "page": page["page"],

        "tables": []
    }

    for table in page["tables"]:

        rows = table["rows"]

        clean_rows = normalize_table(
            rows
        )

        page_out["tables"].append({

            "table_id":
                table["table_id"],

            "rows":
                clean_rows,

            "n_rows":
                len(clean_rows),

            "n_cols":
                max(
                    [len(r) for r in clean_rows],
                    default=0
                )
        })

    normalized.append(page_out)


with open(
    "phase4_normalized.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        normalized,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved phase4_normalized.json")

Saved phase4_normalized.json


In [ ]:
import json
import pandas as pd

with open(
    "phase4_normalized.json",
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

for page in data:

    print("\n")
    print("="*80)
    print("PAGE", page["page"])
    print("="*80)

    for table in page["tables"]:

        print(
            "\nTABLE",
            table["table_id"],
            f"({table['n_rows']} x {table['n_cols']})"
        )

        display(
            pd.DataFrame(
                table["rows"]
            ).head(20)
        )



PAGE 50

TABLE 1 (25 x 8)


,0,1,2,3,4,5,6,7
0,,आं(cid:364) (cid:366)देश / Andhra Pradesh,,2026-27,,,,
1,,Category,,APSPDCL,,APEPDCL,,APCPDCL
2,,HT Category at 11 kV,,,,,,
3,"HT I(B) Townships, Colonies, Gated\nCommunitie...",,0.97,,0.89,,1.12,
4,HT II (A) Commercial& Others,,2.09,,2.20,,2.32,
5,HT II (A) Function Halls/ Auditoriums,,1.99,,1.99,,1.99,
6,HT II (B) Start-up power,,1.99,,1.99,,-,
7,HT III(A) Industry,,1.89,,1.71,,1.85,
8,HT III(B) Seasonal Industries (Off Season),,2.38,,1.76,,1.74,
9,HT III (C ) Energy Intensive Industries,,1.58,,-,,1.75,




PAGE 51

TABLE 1 (57 x 10)


,0,1,2,3,4,5,6,7,8,9
0,HT II(B) Start Up Power,,1.99,,1.99,,,,1.99,
1,HT III(A) Industry,,1.37,,1.44,,,,1.40,
2,HT III(C ) Energy Intensive Industry,,0.31,,0.51,,,,-,
3,HT IV(D) Railway Traction,,1.63,,1.59,,,,1.56,
4,HT V (E ),,1.43,,-,,,,1.43,
5,,220 kV,,,,,,,,
6,HT II(A) Commercial & Others,,-,,1.66,,,,-,
7,HT II(B) Start up Power,,1.99,,1.99,,,,-,
8,HT III(A) Industry,,1.57,,1.37,,,,1.96,
9,HT III (C ) Energy Intensive Industries,,-,,1.57,,,,-,




PAGE 52

TABLE 1 (58 x 13)


...

,0,1,2,3,4,5,6,7,8,9
0,LT-3b,,3.05,,3.05,3.05,,3.05,,3.05
1,LT-4a,,0.49,,0.49,0.49,,0.49,,0.49
2,LT-4b,,1.07,,1.07,1.07,,1.07,,1.07
3,LT-4c,,1.85,,1.85,1.85,,1.85,,1.85
4,LT-5 Industrial,,1.16,,1.16,1.16,,1.16,,1.16
5,LT-7 Temporary Supply,,3.19,,3.19,3.19,,3.19,,3.19
6,,केरल / Kerala,,2026-27,,,,,,
7,,Category,,KSEBL,,,,,,
8,EHT- I,,1.41,,,,,,,
9,EHT- II,,1.34,,,,,,,




PAGE 54

TABLE 1 (54 x 12)


,0,1,2,3,4,5,6,7,8,9,10,11
0,EHT Level,,,,,,,,,,,
1,HT & EHT Ferro Alloys Industrial,,NIL,,,,,,,,,
2,,ओिडशा / Odisha,,2026-27,,,,,,,,
3,,Category,,TPCODL,TPNODL,,,,,,TPSODL,TPWODL
4,EHT,,,1.52,1.43,,,,,,1.18,2.50
5,HT,,,0.56,0.07,,,,,,0.17,1.06
6,LT,,,1.13,0.34,,,,,,0.24,1.40
7,,पुडुचेरी / Puducherry,,2026-27,,,,,,,,
8,,Category,,Puducherry Electricity Department (PED),,,,,,,,
9,LT Level,,NIL,,,,,,,,,




PAGE 55

TABLE 1 (58 x 13)


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,,HT Category 132kV,,,,,,,,,,,
1,HT-I Industry,,1.62,,,,,,1.69,,,,
2,HT-I(B)Ferro Alloys,,-,,,,,,-,,,,
3,HT-II Others (Commercial),,2.34,,,,,,3.77,,,,
4,"HT-III Airports, Bus & Rly Stations",,1.71,,,,,,-,,,,
5,HT-IV(A) Irrigation & Agriculture,,1.60,,,,,,1.92,,,,
6,HT-IV(B) CPWS,,0.76,,,,,,0.58,,,,
7,HT-V(A) Railway Traction,,1.34,,,,,,1.01,,,,
8,HT-V(B)HMR,,0.53,,,,,,-,,,,
9,HT-VI Townships & Residential Colonies,,-,,,,,,1.74,,,,


In [ ]:


import json
import re


with open(
    "phase4_normalized.json",
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)


YEAR_RE = re.compile(
    r"20\d{2}\-\d{2}"
)


def find_state_name(row):

    text = " ".join(
        [str(x) for x in row]
    )

    text = text.strip()

    if not YEAR_RE.search(text):
        return None

    parts = text.split("/")

    if len(parts) > 1:

        state = parts[-1].strip()

        state = YEAR_RE.sub(
            "",
            state
        ).strip()

        return state

    return None


blocks = []

for page in data:

    for table in page["tables"]:

        rows = table["rows"]

        current = None

        for row in rows:

            state = find_state_name(row)

            if state:

                if current:
                    blocks.append(current)

                current = {

                    "page":
                        page["page"],

                    "state":
                        state,

                    "rows": []
                }

                continue

            if current:
                current["rows"].append(row)

        if current:
            blocks.append(current)


with open(
    "state_blocks.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        blocks,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "Detected blocks:",
    len(blocks)
)



for b in blocks[:10]:

    print()
    print("="*60)

    print(
        "STATE:",
        b["state"]
    )

    print(
        "PAGE:",
        b["page"]
    )

    print(
        "ROWS:",
        len(b["rows"])
    )

    for r in b["rows"][:5]:
        print(r)

Detected blocks: 28

STATE: Andhra Pradesh
PAGE: 50
ROWS: 24
['', 'Category', '', 'APSPDCL', '', 'APEPDCL', '', 'APCPDCL']
['', 'HT Category at 11 kV', '', '', '', '', '', '']
['HT I(B) Townships, Colonies, Gated\nCommunities and Villas', '', '0.97', '', '0.89', '', '1.12', '']
['HT II (A) Commercial& Others', '', '2.09', '', '2.20', '', '2.32', '']
['HT II (A) Function Halls/ Auditoriums', '', '1.99', '', '1.99', '', '1.99', '']

STATE: Assam
PAGE: 51
ROWS: 11
['', 'Category', '', 'APDCL', '', '', '', '', '', '']
['HT Domestic and Commercial 30 kW & above', '', '1.81', '', '', '', '', '', '', '']
['Public Water Works', '', '1.81', '', '', '', '', '', '', '']
['Bulk Supply Govt. Edu Inst and others', '', '1.81', '', '', '', '', '', '', '']
['HT Small Industries up to 50 kW', '', '1.81', '', '', '', '', '', '', '']

STATE: Bihar
PAGE: 51
ROWS: 3
['', 'Category', '', 'NBPDCL', '', '', '', 'SBPDCL', '', '']
['', 'HT Category', '', '', '', '', '', '', '', '']
['', 'For 11 kV, 33 kV,132 kV,

In [ ]:


import json
import re



with open(
    "phase4_normalized.json",
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)


YEAR_RE = re.compile(
    r"20\d{2}-\d{2}"
)

def detect_state(row):

    text = " ".join(
        [str(x) for x in row]
    )

    if not YEAR_RE.search(text):
        return None

    if "/" not in text:
        return None

    state = text.split("/")[-1]

    state = YEAR_RE.sub(
        "",
        state
    )

    return state.strip()


stream = []

for page in data:

    page_num = page["page"]

    for table in page["tables"]:

        for row in table["rows"]:

            stream.append({

                "page": page_num,

                "row": row
            })



blocks = []

current = None

for item in stream:

    row = item["row"]

    state = detect_state(row)

    if state:

        if current:

            blocks.append(
                current
            )

        current = {

            "state": state,

            "start_page":
                item["page"],

            "rows": []
        }

        continue

    if current:

        current["rows"].append(
            row
        )

# final block

if current:

    blocks.append(
        current
    )


with open(
    "state_blocks_v2.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        blocks,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "Blocks:",
    len(blocks)
)

for b in blocks[:10]:

    print(
        b["state"],
        "rows:",
        len(b["rows"])
    )

Blocks: 28
Andhra Pradesh rows: 36
Assam rows: 11
Bihar rows: 3
Chandigarh rows: 4
Chhattisgarh rows: 12
Delhi rows: 10
Goa rows: 4
Gujarat rows: 2
Haryana rows: 4
Himachal Pradesh rows: 12


In [ ]:

def parse_simple_matrix_state(block):

    rows = []

    for r in block["rows"]:

        c = compress_row(r)

        if c:
            rows.append(c)

    if len(rows) < 2:

        return {

            "state":
                block["state"],

            "start_page":
                block["start_page"],

            "parser_type":
                "simple_matrix",

            "columns": [],

            "rows": []
        }

    header = rows[0]

    columns = header[1:]

    parsed_rows = []

    for row in rows[1:]:

        entry = {

            "category":
                row[0]
        }

        for col, val in zip(
            columns,
            row[1:]
        ):

            entry[col] = val

        parsed_rows.append(
            entry
        )

    return {

        "state":
            block["state"],

        "start_page":
            block["start_page"],

        "parser_type":
            "simple_matrix",

        "columns":
            columns,

        "rows":
            parsed_rows
    }

In [ ]:
def detect_parser_type(block):

    rows = []

    for r in block["rows"]:

        c = compress_row(r)

        if c:
            rows.append(c)

    if not rows:

        return "unknown"


    header = rows[0]

    if (
        len(header) > 1
        and header[0].lower() == "category"
    ):

        # sectioned matrix

        for r in rows[1:]:

            if len(r) == 1:

                return "matrix"

        # no sections

        return "simple_matrix"


    return "key_value"

In [ ]:
import json

with open(
    "state_blocks_v2.json",
    "r",
    encoding="utf-8"
) as f:

    blocks = json.load(f)

ap = next(
    b for b in blocks
    if b["state"] == "Andhra Pradesh"
)

print(
    "AP rows:",
    len(ap["rows"])
)

for r in ap["rows"][-20:]:

    print(r)

AP rows: 36
['HT III(A) Industry', '', '1.54', '', '1.48', '', '1.55', '']
['HT III(B) Seasonal Industries (Off Season)', '', '2.17', '', '1.53', '', '1.59', '']
['HT III(C ) Energy Intensive Industries', '', '0.36', '', '0.24', '', '-', '']
['HT IV(A) Utilities (Composite Protected Water\nSupply Schemes)', '', '1.76', '', '2.13', '', '-', '']
['HT IV(B) General Purpose', '', '1.90', '', '2.04', '', '1.74', '']
['HT V(E) Government/ Private Lift Irrigation', '', '1.25', '', '1.27', '', '1.25', '']
['', 'HT Category at 132 kV', '', '', '', '', '', '']
['HT II(A) Commercial & Others', '', '1.69', '', '1.99', '', '-', '']
['HT II(B) Start Up Power', '', '1.99', '', '1.99', '', '', '', '1.99', '']
['HT III(A) Industry', '', '1.37', '', '1.44', '', '', '', '1.40', '']
['HT III(C ) Energy Intensive Industry', '', '0.31', '', '0.51', '', '', '', '-', '']
['HT IV(D) Railway Traction', '', '1.63', '', '1.59', '', '', '', '1.56', '']
['HT V (E )', '', '1.43', '', '-', '', '', '', '1.43', '']
[''

In [ ]:
# # ==========================================================
# # PHASE 4.4
# # BUILD STATE CATALOG
# # ==========================================================

# import json
# import pandas as pd

# with open(
#     "state_blocks.json",
#     "r",
#     encoding="utf-8"
# ) as f:
#     blocks = json.load(f)

# catalog = []

# for i, block in enumerate(blocks):

#     text_parts = []

#     for row in block["rows"]:
#         text_parts.extend(
#             [str(x) for x in row]
#         )

#     catalog.append({

#         "block_id": i,

#         "state": block["state"],

#         "page": block["page"],

#         "row_count": len(
#             block["rows"]
#         ),

#         "search_text":
#             " ".join(text_parts)
#     })

# catalog_df = pd.DataFrame(
#     catalog
# )

# catalog_df.to_csv(
#     "state_catalog.csv",
#     index=False
# )

# print(
#     "Total blocks:",
#     len(catalog_df)
# )

# display(
#     catalog_df[
#         [
#             "block_id",
#             "state",
#             "page",
#             "row_count"
#         ]
#     ]
# )

KeyError: 'page'

In [ ]:


import json
import pandas as pd



with open(
    "state_blocks_v2.json",
    "r",
    encoding="utf-8"
) as f:

    blocks = json.load(f)



catalog = []

for block_id, block in enumerate(blocks):

    text_parts = []

    for row in block["rows"]:

        text_parts.extend(
            [str(x) for x in row]
        )

    catalog.append({

        "block_id":
            block_id,

        "state":
            block["state"],

        "start_page":
            block["start_page"],

        "row_count":
            len(block["rows"]),

        "search_text":
            " ".join(text_parts)
    })

catalog_df = pd.DataFrame(
    catalog
)



catalog_df.to_csv(
    "state_catalog_v2.csv",
    index=False
)



print("="*70)
print("STATE CATALOG")
print("="*70)

print(
    "Total States:",
    len(catalog_df)
)

display(
    catalog_df[
        [
            "block_id",
            "state",
            "start_page",
            "row_count"
        ]
    ]
)

STATE CATALOG
Total States: 28


,block_id,state,start_page,row_count
0,0,Andhra Pradesh,50,36
1,1,Assam,51,11
2,2,Bihar,51,3
3,3,Chandigarh,51,4
4,4,Chhattisgarh,51,12
5,5,Delhi,51,10
6,6,Goa,52,4
7,7,Gujarat,52,2
8,8,Haryana,52,4
9,9,Himachal Pradesh,52,12


In [ ]:


import json
import pandas as pd

catalog = pd.read_csv(
    "state_catalog.csv"
)

with open(
    "state_blocks_v2.json",
    "r",
    encoding="utf-8"
) as f:
    blocks = json.load(f)

def retrieve_state(
    state_name
):

    hits = catalog[
        catalog["state"]
        .str.contains(
            state_name,
            case=False,
            na=False
        )
    ]

    if len(hits) == 0:

        return None

    block_id = int(
        hits.iloc[0]["block_id"]
    )

    return blocks[
        block_id
    ]



for state in [

    "Andhra Pradesh",
    "Goa",
    "Haryana",
    "Delhi"

]:

    result = retrieve_state(
        state
    )

    print()
    print("="*60)
    print(state)
    print("="*60)

    print(
        "Rows:",
        len(result["rows"])
    )

    for r in result["rows"][:10]:
        print(r)


Andhra Pradesh
Rows: 36
['', 'Category', '', 'APSPDCL', '', 'APEPDCL', '', 'APCPDCL']
['', 'HT Category at 11 kV', '', '', '', '', '', '']
['HT I(B) Townships, Colonies, Gated\nCommunities and Villas', '', '0.97', '', '0.89', '', '1.12', '']
['HT II (A) Commercial& Others', '', '2.09', '', '2.20', '', '2.32', '']
['HT II (A) Function Halls/ Auditoriums', '', '1.99', '', '1.99', '', '1.99', '']
['HT II (B) Start-up power', '', '1.99', '', '1.99', '', '-', '']
['HT III(A) Industry', '', '1.89', '', '1.71', '', '1.85', '']
['HT III(B) Seasonal Industries (Off Season)', '', '2.38', '', '1.76', '', '1.74', '']
['HT III (C ) Energy Intensive Industries', '', '1.58', '', '-', '', '1.75', '']
['HT IV (A) Utilities (Composite Protected Water\nSupply Schemes)', '', '1.86', '', '1.91', '', '1.80', '']

Goa
Rows: 4
['', 'Category', '', 'Electricity Department, Goa (EDG)', '', '', '', '', '', '', '', '', '']
['Low Tension (LT) Level', '', 'NIL', '', '', '', '', '', '', '', '', '', '']
['High Tensi

In [ ]:


import pandas as pd

catalog = pd.read_csv(
    "state_catalog_v2.csv"
)

ALL_STATES = sorted(
    catalog["state"].unique()
)

def detect_state(query):

    q = query.lower()

    for state in ALL_STATES:

        if state.lower() in q:

            return state

    return None


tests = [

    "Extract Goa Open Access Charges",

    "Show Haryana Open Access Charges",

    "Get Andhra Pradesh Charges",

    "Delhi Open Access Charges",

    "What is the OA charge in Gujarat"
]

for t in tests:

    print()
    print("QUERY :", t)

    print(
        "STATE :",
        detect_state(t)
    )


QUERY : Extract Goa Open Access Charges
STATE : Goa

QUERY : Show Haryana Open Access Charges
STATE : Haryana

QUERY : Get Andhra Pradesh Charges
STATE : Andhra Pradesh

QUERY : Delhi Open Access Charges
STATE : Delhi

QUERY : What is the OA charge in Gujarat
STATE : Gujarat


In [ ]:


import json
import pandas as pd

catalog = pd.read_csv(
    "state_catalog_v2.csv"
)

with open(
    "state_blocks_v2.json",
    "r",
    encoding="utf-8"
) as f:

    blocks = json.load(f)

def detect_state(query):

    q = query.lower()

    for state in catalog["state"]:

        if state.lower() in q:

            return state

    return None

def retrieve_block(query):

    state = detect_state(query)

    if state is None:

        return None

    hit = catalog[
        catalog["state"] == state
    ].iloc[0]

    return blocks[
        int(hit["block_id"])
    ]



query = (
    "Extract Andhra Pradesh Open Access Charges"
)

result = retrieve_block(
    query
)

print(
    json.dumps(
        result,
        indent=2,
        ensure_ascii=False
    )
)

{
  "state": "Andhra Pradesh",
  "start_page": 50,
  "rows": [
    [
      "",
      "Category",
      "",
      "APSPDCL",
      "",
      "APEPDCL",
      "",
      "APCPDCL"
    ],
    [
      "",
      "HT Category at 11 kV",
      "",
      "",
      "",
      "",
      "",
      ""
    ],
    [
      "HT I(B) Townships, Colonies, Gated\nCommunities and Villas",
      "",
      "0.97",
      "",
      "0.89",
      "",
      "1.12",
      ""
    ],
    [
      "HT II (A) Commercial& Others",
      "",
      "2.09",
      "",
      "2.20",
      "",
      "2.32",
      ""
    ],
    [
      "HT II (A) Function Halls/ Auditoriums",
      "",
      "1.99",
      "",
      "1.99",
      "",
      "1.99",
      ""
    ],
    [
      "HT II (B) Start-up power",
      "",
      "1.99",
      "",
      "1.99",
      "",
      "-",
      ""
    ],
    [
      "HT III(A) Industry",
      "",
      "1.89",
      "",
      "1.71",
      "",
      "1.85",
      ""
    ],
    [
      "HT III(B)

In [ ]:


import json

with open(
    "state_blocks_v2.json",
    "r",
    encoding="utf-8"
) as f:
    blocks = json.load(f)



goa = None

for b in blocks:

    if b["state"] == "Goa":

        goa = b
        break


def compress_row(row):

    vals = []

    for x in row:

        x = str(x).strip()

        if x != "":
            vals.append(x)

    return vals



compressed = []

for row in goa["rows"]:

    r = compress_row(row)

    if len(r) > 0:
        compressed.append(r)

print(
    json.dumps(
        compressed,
        indent=2,
        ensure_ascii=False
    )
)

[
  [
    "Category",
    "Electricity Department, Goa (EDG)"
  ],
  [
    "Low Tension (LT) Level",
    "NIL"
  ],
  [
    "High Tension (HT)/ EHT Level",
    "1.13"
  ],
  [
    "Extra High Tension (EHT) Level",
    "0.68"
  ]
]


In [ ]:


import json

with open(
    "state_blocks_v2.json",
    "r",
    encoding="utf-8"
) as f:
    blocks = json.load(f)

STATE = "Haryana"

block = None

for b in blocks:
    if b["state"] == STATE:
        block = b
        break

def compress_row(row):

    return [
        str(x).strip()
        for x in row
        if str(x).strip() != ""
    ]

compressed = []

for row in block["rows"]:

    r = compress_row(row)

    if r:
        compressed.append(r)

print(
    json.dumps(
        compressed[:20],
        indent=2,
        ensure_ascii=False
    )
)

[
  [
    "Category",
    "UHBVNL",
    "DHBVNL"
  ],
  [
    "HT Supply",
    "1.45",
    "1.45"
  ],
  [
    "Bulk Supply (other than DS)",
    "-",
    "-"
  ],
  [
    "LT Supply",
    "1.18",
    "1.18"
  ]
]


In [ ]:


import json

with open(
    "state_blocks_v2.json",
    "r",
    encoding="utf-8"
) as f:
    blocks = json.load(f)



block = next(
    b for b in blocks
    if b["state"] == "Andhra Pradesh"
)


def compress_row(row):

    return [
        str(x).strip()
        for x in row
        if str(x).strip() != ""
    ]

rows = []

for r in block["rows"]:

    c = compress_row(r)

    if c:
        rows.append(c)



columns = None

sections = []

current_section = None

for row in rows:

    # header row

    if (
        len(row) > 1
        and row[0].lower() == "category"
    ):

        columns = row[1:]

        continue

    # section row

    if len(row) == 1:

        current_section = {

            "name": row[0],

            "columns": columns,

            "rows": []
        }

        sections.append(
            current_section
        )

        continue

    # data row

    if current_section:

        entry = {

            "category": row[0]
        }

        for col, val in zip(
            columns,
            row[1:]
        ):

            entry[col] = val

        current_section["rows"].append(
            entry
        )



result = {

    "state":
        "Andhra Pradesh",

    "sections":
        sections
}

print(
    json.dumps(
        result,
        indent=2,
        ensure_ascii=False
    )
)

{
  "state": "Andhra Pradesh",
  "sections": [
    {
      "name": "HT Category at 11 kV",
      "columns": [
        "APSPDCL",
        "APEPDCL",
        "APCPDCL"
      ],
      "rows": [
        {
          "category": "HT I(B) Townships, Colonies, Gated\nCommunities and Villas",
          "APSPDCL": "0.97",
          "APEPDCL": "0.89",
          "APCPDCL": "1.12"
        },
        {
          "category": "HT II (A) Commercial& Others",
          "APSPDCL": "2.09",
          "APEPDCL": "2.20",
          "APCPDCL": "2.32"
        },
        {
          "category": "HT II (A) Function Halls/ Auditoriums",
          "APSPDCL": "1.99",
          "APEPDCL": "1.99",
          "APCPDCL": "1.99"
        },
        {
          "category": "HT II (B) Start-up power",
          "APSPDCL": "1.99",
          "APEPDCL": "1.99",
          "APCPDCL": "-"
        },
        {
          "category": "HT III(A) Industry",
          "APSPDCL": "1.89",
          "APEPDCL": "1.71",
          "APCPDCL": 

In [ ]:


import json



with open(
    "state_blocks_v2.json",
    "r",
    encoding="utf-8"
) as f:

    blocks = json.load(f)



def compress_row(row):

    return [

        str(x).strip()

        for x in row

        if str(x).strip() != ""
    ]



def parse_matrix_state(block):

    rows = []

    for r in block["rows"]:

        c = compress_row(r)

        if c:
            rows.append(c)

    columns = None

    sections = []

    current_section = None

    for row in rows:

        # header

        if (

            len(row) > 1

            and row[0].lower() == "category"

        ):

            columns = row[1:]

            continue

        # section

        if len(row) == 1:

            current_section = {

                "name":
                    row[0],

                "columns":
                    columns,

                "rows": []
            }

            sections.append(
                current_section
            )

            continue

        # data row

        if current_section and columns:

            entry = {

                "category":
                    row[0]
            }

            for col, val in zip(

                columns,

                row[1:]
            ):

                entry[col] = val

            current_section[
                "rows"
            ].append(entry)

    return {

        "state":
            block["state"],

        "start_page":
            block["start_page"],

        "parser_type":
            "matrix",

        "sections":
            sections
    }


def parse_simple_matrix_state(block):

    rows = []

    for r in block["rows"]:

        c = compress_row(r)

        if c:
            rows.append(c)

    if len(rows) < 2:

        return {

            "state":
                block["state"],

            "start_page":
                block["start_page"],

            "parser_type":
                "simple_matrix",

            "columns": [],

            "rows": []
        }

    header = rows[0]

    columns = header[1:]

    parsed_rows = []

    for row in rows[1:]:

        entry = {

            "category":
                row[0]
        }

        for col, val in zip(

            columns,

            row[1:]
        ):

            entry[col] = val

        parsed_rows.append(
            entry
        )

    return {

        "state":
            block["state"],

        "start_page":
            block["start_page"],

        "parser_type":
            "simple_matrix",

        "columns":
            columns,

        "rows":
            parsed_rows
    }



def parse_key_value_state(block):

    rows = []

    for r in block["rows"]:

        c = compress_row(r)

        if c:
            rows.append(c)

    utility = None

    data = []

    for row in rows:

        if (

            len(row) >= 2

            and row[0].lower() == "category"

        ):

            utility = row[1]

            continue

        if len(row) >= 2:

            data.append({

                "metric":
                    row[0],

                "value":
                    row[1]
            })

    return {

        "state":
            block["state"],

        "start_page":
            block["start_page"],

        "parser_type":
            "key_value",

        "utility":
            utility,

        "data":
            data
    }



def detect_parser_type(block):

    rows = []

    for r in block["rows"]:

        c = compress_row(r)

        if c:
            rows.append(c)

    if not rows:

        return "unknown"

    header = rows[0]


    if (

        len(header) > 1

        and header[0].lower() == "category"

    ):

        # look for section rows

        for row in rows[1:]:

            if len(row) == 1:

                return "matrix"

        return "simple_matrix"

    return "key_value"



structured_states = []

for block in blocks:

    parser_type = detect_parser_type(
        block
    )

    try:

        if parser_type == "matrix":

            parsed = parse_matrix_state(
                block
            )

        elif parser_type == "simple_matrix":

            parsed = parse_simple_matrix_state(
                block
            )

        else:

            parsed = parse_key_value_state(
                block
            )

        structured_states.append(
            parsed
        )

    except Exception as e:

        structured_states.append({

            "state":
                block["state"],

            "start_page":
                block["start_page"],

            "parser_type":
                "failed",

            "error":
                str(e)
        })



with open(
    "structured_states_v2.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(

        structured_states,

        f,

        ensure_ascii=False,

        indent=2
    )



print("="*70)
print("PARSING SUMMARY")
print("="*70)

for s in structured_states:

    print(
        f"{s['state']:25s}",
        s["parser_type"]
    )

print()
print(
    "Total States:",
    len(structured_states)
)

print()
print(
    "Saved -> structured_states_v2.json"
)

PARSING SUMMARY
Andhra Pradesh            matrix
Assam                     simple_matrix
Bihar                     matrix
Chandigarh                simple_matrix
Chhattisgarh              matrix
Delhi                     matrix
Goa                       simple_matrix
Gujarat                   simple_matrix
Haryana                   simple_matrix
Himachal Pradesh          matrix
Jammu & Kashmir           simple_matrix
Jharkhand                 simple_matrix
Karnataka                 matrix
Kerala                    simple_matrix
Madhya Pradesh            simple_matrix
Maharashtra               matrix
Meghalaya                 matrix
Odisha                    simple_matrix
Puducherry                simple_matrix
Punjab                    simple_matrix
Rajasthan                 matrix
Sikkim                    simple_matrix
Tamil Nadu                simple_matrix
Telangana                 matrix
Tripura                   matrix
Uttar Pradesh             simple_matrix
Uttarakhand          

In [ ]:


import json

with open(
    "structured_states_v2.json",
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

for state_name in [

    "Andhra Pradesh",
    "Assam",
    "Goa",
    "Delhi",
    "Telangana"

]:

    print("\n")
    print("="*80)
    print(state_name)
    print("="*80)

    obj = next(

        x for x in data

        if x["state"] == state_name
    )

    print(

        json.dumps(

            obj,

            indent=2,

            ensure_ascii=False

        )[:5000]
    )



Andhra Pradesh
{
  "state": "Andhra Pradesh",
  "start_page": 50,
  "parser_type": "matrix",
  "sections": [
    {
      "name": "HT Category at 11 kV",
      "columns": [
        "APSPDCL",
        "APEPDCL",
        "APCPDCL"
      ],
      "rows": [
        {
          "category": "HT I(B) Townships, Colonies, Gated\nCommunities and Villas",
          "APSPDCL": "0.97",
          "APEPDCL": "0.89",
          "APCPDCL": "1.12"
        },
        {
          "category": "HT II (A) Commercial& Others",
          "APSPDCL": "2.09",
          "APEPDCL": "2.20",
          "APCPDCL": "2.32"
        },
        {
          "category": "HT II (A) Function Halls/ Auditoriums",
          "APSPDCL": "1.99",
          "APEPDCL": "1.99",
          "APCPDCL": "1.99"
        },
        {
          "category": "HT II (B) Start-up power",
          "APSPDCL": "1.99",
          "APEPDCL": "1.99",
          "APCPDCL": "-"
        },
        {
          "category": "HT III(A) Industry",
          "APSP

In [ ]:


import json
from openpyxl import Workbook
from openpyxl.styles import Font

with open(
    "structured_states_v2.json",
    "r",
    encoding="utf-8"
) as f:

    states = json.load(f)

wb = Workbook()

wb.remove(
    wb.active
)

for state in states:

    ws = wb.create_sheet(
        title=state["state"][:31]
    )

    row_ptr = 1

    ws.cell(
        row=row_ptr,
        column=1
    ).value = state["state"]

    ws.cell(
        row=row_ptr,
        column=1
    ).font = Font(
        bold=True,
        size=14
    )

    row_ptr += 2



    if state["parser_type"] == "matrix":

        for section in state["sections"]:

            ws.cell(
                row=row_ptr,
                column=1
            ).value = section["name"]

            ws.cell(
                row=row_ptr,
                column=1
            ).font = Font(
                bold=True
            )

            row_ptr += 1

            columns = (
                ["category"]
                + section["columns"]
            )

            for c_idx, col in enumerate(
                columns,
                start=1
            ):

                ws.cell(
                    row=row_ptr,
                    column=c_idx
                ).value = col

                ws.cell(
                    row=row_ptr,
                    column=c_idx
                ).font = Font(
                    bold=True
                )

            row_ptr += 1

            for record in section["rows"]:

                for c_idx, col in enumerate(
                    columns,
                    start=1
                ):

                    ws.cell(
                        row=row_ptr,
                        column=c_idx
                    ).value = record.get(
                        col,
                        ""
                    )

                row_ptr += 1

            row_ptr += 2



    elif state["parser_type"] == "simple_matrix":

        columns = (
            ["category"]
            + state["columns"]
        )

        for c_idx, col in enumerate(
            columns,
            start=1
        ):

            ws.cell(
                row=row_ptr,
                column=c_idx
            ).value = col

            ws.cell(
                row=row_ptr,
                column=c_idx
            ).font = Font(
                bold=True
            )

        row_ptr += 1

        for record in state["rows"]:

            for c_idx, col in enumerate(
                columns,
                start=1
            ):

                ws.cell(
                    row=row_ptr,
                    column=c_idx
                ).value = record.get(
                    col,
                    ""
                )

            row_ptr += 1



    elif state["parser_type"] == "key_value":

        ws.cell(
            row=row_ptr,
            column=1
        ).value = "Utility"

        ws.cell(
            row=row_ptr,
            column=2
        ).value = state.get(
            "utility",
            ""
        )

        row_ptr += 2

        ws.cell(
            row=row_ptr,
            column=1
        ).value = "Metric"

        ws.cell(
            row=row_ptr,
            column=2
        ).value = "Value"

        row_ptr += 1

        for item in state["data"]:

            ws.cell(
                row=row_ptr,
                column=1
            ).value = item["metric"]

            ws.cell(
                row=row_ptr,
                column=2
            ).value = item["value"]

            row_ptr += 1


    for column in ws.columns:

        max_len = 0

        letter = (
            column[0]
            .column_letter
        )

        for cell in column:

            try:

                max_len = max(
                    max_len,
                    len(str(cell.value))
                )

            except:
                pass

        ws.column_dimensions[
            letter
        ].width = min(
            max_len + 2,
            60
        )

wb.save(
    "Open_Access_Charges_All_States.xlsx"
)

print(
    "Saved: Open_Access_Charges_All_States.xlsx"
)

Saved: Open_Access_Charges_All_States.xlsx


In [ ]:
# import json

# with open(
#     "state_blocks_v2.json",
#     "r",
#     encoding="utf-8"
# ) as f:
#     blocks = json.load(f)

# for target in [

#     "Gujarat",
#     "Haryana",
#     "Jammu & Kashmir",
#     "Madhya Pradesh",
#     "Odisha"

# ]:

#     block = next(
#         b for b in blocks
#         if b["state"] == target
#     )

#     print("\n")
#     print("="*80)
#     print(target)
#     print("="*80)

#     for row in block["rows"][:15]:

#         compressed = [

#             str(x).strip()

#             for x in row

#             if str(x).strip() != ""
#         ]

#         print(compressed)



Gujarat
['Category', 'DGVCL', 'PGVCL', 'MGVCL', 'UGVCL']
['HT Category', '1.33', '1.33', '1.33', '1.33']


Haryana
['Category', 'UHBVNL', 'DHBVNL']
['HT Supply', '1.45', '1.45']
['Bulk Supply (other than DS)', '-', '-']
['LT Supply', '1.18', '1.18']


Jammu & Kashmir
['Category', 'JPDCL', 'KPDCL']
['HT Consumers', '1.38/kWh (or 1.24/kVAh)', '1.38/kWh (or 1.24/kVAh)']


Madhya Pradesh
['Category', 'MPMaKVVCL', 'MPPaKVV', 'MPPoKVVCL']
['LV-1: Domestic', '0.67', 'CL\n0.67', '0.67']
['LV-2 & 4: Non-Domestic & Industrial', '1.49', '1.49', '1.49']
['LV-3: Public Water Works & Street Light', '0.77', '0.77', '0.77']
['LV-6: E-Vehicle/ E-Rickshaws Charging Stations', '1.02', '1.02', '1.02']
['HV-1: Railway Traction', '1.49', '1.49', '1.49']
['HV- 2: Coal Mines', '1.49', '1.49', '1.49']
['HV- 3.1 & 3.2: Industrial and Non-Industrial', '1.49', '1.49', '1.49']
['HV- 3.3: Shopping Malls', '1.49', '1.49', '1.49']
['HV- 3.4: Power Intensive Industries', '0.90', '0.90', '0.90']
['HV- 4: Seasonal & N

In [ ]:
# ==========================================================
# PHASE 6.1
# LOAD STRUCTURED DATA
# ==========================================================

import json
import re

# ==========================================================
# LOAD
# ==========================================================

with open(
    "structured_states_v2.json",
    "r",
    encoding="utf-8"
) as f:

    STATES = json.load(f)

# ==========================================================
# STATE LIST
# ==========================================================

STATE_NAMES = sorted(

    [x["state"] for x in STATES],

    key=len,

    reverse=True
)

# ==========================================================
# DETECT STATE
# ==========================================================

def detect_state(query):

    q = query.lower()

    for state in STATE_NAMES:

        if state.lower() in q:

            return state

    return None

# ==========================================================
# TEST
# ==========================================================

tests = [

    "Extract Andhra Pradesh Industry Charges",

    "Show Haryana LT Supply",

    "Find Goa HT Level",

    "Delhi Non Domestic",

    "Odisha EHT Charges"
]

for t in tests:

    print()
    print(t)

    print(
        "STATE:",
        detect_state(t)
    )


Extract Andhra Pradesh Industry Charges
STATE: Andhra Pradesh

Show Haryana LT Supply
STATE: Haryana

Find Goa HT Level
STATE: Goa

Delhi Non Domestic
STATE: Delhi

Odisha EHT Charges
STATE: Odisha


In [ ]:
# ==========================================================
# PHASE 6.2
# KEYWORD EXTRACTION
# ==========================================================

STOP_WORDS = {

    "extract",
    "show",
    "find",
    "get",
    "what",
    "is",
    "the",
    "charges",
    "charge",
    "open",
    "access",
    "for",
    "of",
    "in"
}

def extract_keywords(query):

    state = detect_state(query)

    q = query.lower()

    if state:

        q = q.replace(
            state.lower(),
            ""
        )

    words = re.findall(
        r"[a-zA-Z0-9]+",
        q
    )

    keywords = []

    for w in words:

        if w not in STOP_WORDS:

            keywords.append(w)

    return keywords

# ==========================================================
# TEST
# ==========================================================

tests = [

    "Extract Andhra Pradesh Industry Charges",

    "Show Haryana LT Supply",

    "Find Goa HT Level",

    "Delhi Non Domestic",

    "Odisha EHT Charges"
]

for t in tests:

    print()
    print(t)

    print(
        extract_keywords(t)
    )


Extract Andhra Pradesh Industry Charges
['industry']

Show Haryana LT Supply
['lt', 'supply']

Find Goa HT Level
['ht', 'level']

Delhi Non Domestic
['non', 'domestic']

Odisha EHT Charges
['eht']


In [ ]:
# ==========================================================
# PHASE 6.3 FINAL
# IMPROVED SEARCH ENGINE
# ==========================================================

import re

# ==========================================================
# NORMALIZATION
# ==========================================================

def normalize_text(text):

    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9]+",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

# ==========================================================
# SCORING
# ==========================================================

def score_text(text, keywords):

    text = normalize_text(text)

    score = 0

    for kw in keywords:

        kw = normalize_text(kw)

        if kw in text:

            score += 1

    return score

# ==========================================================
# SEARCH STATE
# ==========================================================

def search_state(state_name, keywords):

    state = next(

        s for s in STATES

        if s["state"] == state_name
    )

    matches = []

    # ======================================================
    # MATRIX STATES
    # ======================================================

    if state["parser_type"] == "matrix":

        for section in state["sections"]:

            section_score = score_text(

                section["name"],

                keywords
            )

            for row in section["rows"]:

                category_score = score_text(

                    row["category"],

                    keywords
                )

                total_score = (

                    section_score
                    +
                    category_score
                )

                if total_score > 0:

                    matches.append({

                        "section":
                            section["name"],

                        "score":
                            total_score,

                        "data":
                            row
                    })

    # ======================================================
    # SIMPLE MATRIX
    # ======================================================

    elif state["parser_type"] == "simple_matrix":

        for row in state["rows"]:

            score = score_text(

                row["category"],

                keywords
            )

            if score > 0:

                matches.append({

                    "score":
                        score,

                    "data":
                        row
                })

    # ======================================================
    # KEY VALUE
    # ======================================================

    elif state["parser_type"] == "key_value":

        for row in state["data"]:

            score = score_text(

                row["metric"],

                keywords
            )

            if score > 0:

                matches.append({

                    "score":
                        score,

                    "data":
                        row
                })

    # ======================================================
    # SORT
    # ======================================================

    matches = sorted(

        matches,

        key=lambda x: x["score"],

        reverse=True
    )

    return matches

In [ ]:
# ==========================================================
# PHASE 6.4 FINAL
# QUERY ENGINE
# ==========================================================

import json

# ==========================================================
# RUN QUERY
# ==========================================================

def run_query(query):

    state = detect_state(
        query
    )

    if state is None:

        return {

            "success": False,

            "message":
                "State not detected"
        }

    keywords = extract_keywords(
        query
    )

    matches = search_state(

        state,

        keywords
    )

    return {

        "success": True,

        "query":
            query,

        "state":
            state,

        "keywords":
            keywords,

        "match_count":
            len(matches),

        "matches":
            matches[:20]
    }

# ==========================================================
# PRETTY PRINT
# ==========================================================

def answer_query(query):

    result = run_query(
        query
    )

    print()
    print("="*80)

    print(
        "QUERY:",
        query
    )

    print("="*80)

    print(
        json.dumps(
            result,
            indent=2,
            ensure_ascii=False
        )[:10000]
    )

    return result

In [ ]:
# ==========================================================
# PHASE 6.5
# RETRIEVAL TEST SUITE
# ==========================================================

queries = [

    "Extract Andhra Pradesh Industry Charges",

    "Andhra Pradesh 220 kV Industry",

    "Andhra Pradesh Railway Traction",

    "Delhi Non Domestic",

    "Delhi DMRC",

    "Telangana Ferro Alloys",

    "Goa HT Level",

    "Goa EHT Level",

    "Haryana LT Supply",

    "Odisha EHT",

    "Odisha HT",

    "Madhya Pradesh Railway Traction",

    "Punjab Bulk Supply",

    "Tamil Nadu HT Industry"
]

for q in queries:

    answer_query(q)


QUERY: Extract Andhra Pradesh Industry Charges
{
  "success": true,
  "query": "Extract Andhra Pradesh Industry Charges",
  "state": "Andhra Pradesh",
  "keywords": [
    "industry"
  ],
  "match_count": 5,
  "matches": [
    {
      "section": "HT Category at 11 kV",
      "score": 1,
      "data": {
        "category": "HT III(A) Industry",
        "APSPDCL": "1.89",
        "APEPDCL": "1.71",
        "APCPDCL": "1.85"
      }
    },
    {
      "section": "HT Category at 33 kV",
      "score": 1,
      "data": {
        "category": "HT III(A) Industry",
        "APSPDCL": "1.54",
        "APEPDCL": "1.48",
        "APCPDCL": "1.55"
      }
    },
    {
      "section": "HT Category at 132 kV",
      "score": 1,
      "data": {
        "category": "HT III(A) Industry",
        "APSPDCL": "1.37",
        "APEPDCL": "1.44",
        "APCPDCL": "1.40"
      }
    },
    {
      "section": "HT Category at 132 kV",
      "score": 1,
      "data": {
        "category": "HT III(C ) Energy I

In [ ]:
while True:

    q = input(
        "\nEnter Query (or exit): "
    )

    if q.lower() == "exit":

        break

    answer_query(q)


Enter Query (or exit): Karnataka Industry Rate

QUERY: Karnataka Industry Rate
{
  "success": true,
  "query": "Karnataka Industry Rate",
  "state": "Karnataka",
  "keywords": [
    "industry",
    "rate"
  ],
  "match_count": 0,
  "matches": []
}

Enter Query (or exit): Karnataka Industry

QUERY: Karnataka Industry
{
  "success": true,
  "query": "Karnataka Industry",
  "state": "Karnataka",
  "keywords": [
    "industry"
  ],
  "match_count": 0,
  "matches": []
}

Enter Query (or exit): Tamil Nadu Industry

QUERY: Tamil Nadu Industry
{
  "success": true,
  "query": "Tamil Nadu Industry",
  "state": "Tamil Nadu",
  "keywords": [
    "industry"
  ],
  "match_count": 1,
  "matches": [
    {
      "score": 1,
      "data": {
        "category": "HTI A: HT-Industry",
        "TANGEDCO": "1.99"
      }
    }
  ]
}

Enter Query (or exit): Delhi Non Domestic

QUERY: Delhi Non Domestic
{
  "success": true,
  "query": "Delhi Non Domestic",
  "state": "Delhi",
  "keywords": [
    "non",
    "d